# Ruling 32's matcher: what a crossing pays, measured

A `mask` secret is stored as, per form of its value, a length `L`, a keyed filter `F` of its first `k` UTF-16 units and
a slow confirm hash `C`; never the value. A crossing slides a window of `k` units over the text with a rolling keyed
hash modulo a prime `p`, looks each position up, and slow-confirms each hit over `L` units. The spec's provisional
widths: `k = 8`, `w = 24` (`p` = 16777213, the largest prime under 2^24), and PBKDF2-SHA256 as the slow hash at about a
millisecond. The questions, at 64 entries on a 1 MiB result:

1. How many forms does a secret really give, and so how many false candidates does a crossing raise?
2. What does the filter pass cost, against today's plaintext scrub over the same text?
3. What does the slow confirm cost at a few iteration counts, and so the whole crossing?
4. Is one filter hit (a text holding a stored 8-unit prefix) larger than the pass's own spread, i.e. a timing oracle?

`measure-matcher.mjs` runs the matcher in Node with the forms from `valueForms` (src/secret-registry.ts), a seeded
generator, and a corpus shaped like tool output (log lines, hex ids, base64 blobs) with three true occurrences planted.
One machine, run alone; the pass is repeated 15 times and the confirm 30.

In [1]:
import json
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "secrets"
DATA = HERE / "data" / "mask-matcher"
D = json.loads((DATA / "matcher.json").read_text())
pd.set_option("display.width", 200)
print(pd.DataFrame([{"file": "data/mask-matcher/matcher.json", "sha256": wmlnb.sha256_file(DATA / "matcher.json")[:16]}]).to_string(index=False))
print(f"{D['cpu']}, node {D['node']}; N = {D['N']} units, k = {D['K']}, p = {D['P']}, {D['repeats']} repeats of each pass")
print(f"load average (1, 5, 15 min) before {[round(x, 1) for x in D['loadBefore']]}, after {[round(x, 1) for x in D['loadAfter']]}: a shared machine")

                          file           sha256
data/mask-matcher/matcher.json 9d8de643774b7621
Apple M5 Pro, node v26.12.1; N = 1048576 units, k = 8, p = 16777213, 15 repeats of each pass
load average (1, 5, 15 min) before [10.6, 7.3, 5.9], after [10.1, 7.7, 6.1]: a shared machine


## 1. Forms, and the false candidates they raise

`forms` is what `valueForms` gives after removing duplicates: for a token of letters, digits and `/+_-`, the JSON, the
percent and the shell forms mostly equal the value, so a secret gives about five forms, not the eleven the spec
assumed. `expectedFalse` is `N·E/p`; `falseCandidates` is one run's count, a Poisson draw around it.

In [2]:
cfg = pd.DataFrame(D["configs"])
print(cfg[["entries", "forms", "refused", "maxL", "candidates", "truePositives", "planted", "falseCandidates", "expectedFalse"]].round(2).to_string(index=False))
assert (cfg["truePositives"] == cfg["planted"]).all(), "the pass missed a planted occurrence"
assert (cfg["refused"] == 0).all()

 entries  forms  refused  maxL  candidates  truePositives  planted  falseCandidates  expectedFalse
       1      4        0    36           3              3        3                0           0.25
       8     39        0    53           6              3        3                3           2.44
      64    311        0    52          13              3        3               10          19.44


## 1b. Thirty fresh keys per kind of text

One run's count is one draw. Here each kind of text gets 30 trials, each with new secrets, a new base for the rolling
hash and a new text. A candidate is a (position, form) pair whose filter matches where the form is not; its model is
`N·E/p`. A hit position is a position with at least one; its model is `N·T/p`, `T` being the number of distinct filters
in the table, which is below `E` because forms of one secret often share their first `k` units (the value and its
JSON or shell form, the two base64 alphabets). `D` counts the different `k`-unit windows in the text, and a distinct hit window is one of
them that hit; its model is `D·T/p`. If clumping is what spreads the pairs, the distinct hit windows are Poisson
(`sd / sqrt(mean)` near 1) on every kind of text while the pairs are not.

In [3]:
tr = pd.DataFrame(D["trials"])
N_pos = tr["positions"].iloc[0]
g = tr.groupby("kind", sort=False)
n = g.size()
s = pd.DataFrame({
    "E": g["E"].mean(), "T": g["T"].mean(),
    "pairs mean": g["falsePairs"].mean(), "pairs sd": g["falsePairs"].std(ddof=1),
    "model N·E/p": g["E"].mean() * N_pos / D["P"],
    "pairs max": g["falsePairs"].max(),
})
s["pairs se"] = s["pairs sd"] / n.pow(0.5)
s["(mean - model) / se"] = (s["pairs mean"] - s["model N·E/p"]) / s["pairs se"]
s["pairs sd / sqrt(mean)"] = s["pairs sd"] / s["pairs mean"].pow(0.5)
print(s.round(2).to_string())
print()
tr["model D·T/p"] = tr["distinctWindows"] * tr["T"] / D["P"]
w = pd.DataFrame({
    "D mean": g["distinctWindows"].mean(), "D / N": g["distinctWindows"].mean() / N_pos,
    "hit windows mean": g["distinctHitWindows"].mean(), "hit windows sd": g["distinctHitWindows"].std(ddof=1),
    "model D·T/p": tr.groupby("kind", sort=False)["model D·T/p"].mean(),
})
w["hit windows se"] = w["hit windows sd"] / n.pow(0.5)
w["(mean - model) / se"] = (w["hit windows mean"] - w["model D·T/p"]) / w["hit windows se"]
w["hit windows sd / sqrt(mean)"] = w["hit windows sd"] / w["hit windows mean"].pow(0.5)
print(w.round(3).to_string())

             E       T  pairs mean  pairs sd  model N·E/p  pairs max  pairs se  (mean - model) / se  pairs sd / sqrt(mean)
kind                                                                                                                      
random  312.83  277.67       19.97      6.12        19.55         36      1.12                 0.37                   1.37
base64  312.83  277.37       18.63      4.89        19.55         32      0.89                -1.03                   1.13
json    314.30  277.73       13.57     16.18        19.64         87      2.95                -2.06                   4.39
log     317.43  279.23       17.40      9.02        19.84         46      1.65                -1.48                   2.16

             D mean  D / N  hit windows mean  hit windows sd  model D·T/p  hit windows se  (mean - model) / se  hit windows sd / sqrt(mean)
kind                                                                                                                     

The expectation does not move with the text: over the random key, `E[pairs] = N·E/p` by linearity, whatever the text
repeats. What repetition changes is the VARIANCE. JSON and the log have far fewer distinct windows than positions (`D /
N`), so a filter that collides with a frequent window hits once per occurrence, and the pair count is a sum of
clumps: right-skewed, mostly low, now and then large (`pairs max`). A sample mean of 30 such draws sits below the
expectation more often than above it, so a count a couple of standard errors low (`(mean - model) / se`) is what
skew gives, not a non-uniform hash. The second table tests that reading: counted by distinct window, the hits sit on
`D·T/p` with Poisson spread on all four texts. The single run in section 1, on the log text, was one such draw. Two
consequences for the build:

- **Memoise the confirm per distinct window within a crossing.** The same `L` units give the same confirm answer, so a
  clump of one repeated window costs one confirm, not one per occurrence. That bounds a crossing's confirms by the
  number of distinct colliding windows, which is what the model counts.
- **Repetitive text is the slow case, not the fast one.** A crossing's worst cost comes from a frequent window that
  collides, so a time budget is set from the tail, not the mean.

## 2. The filter pass, against the plaintext scrub

`pass` looks every position up in a `Map`; `bits` checks a 2 MiB bitset of all `p` filter values first and goes to the
`Map` only on a set bit. `plaintext` is #1948's `scrubRegistry` over the same text with the same secrets as values.

In [4]:
base = pd.DataFrame(D["baseline"])
t = cfg[["entries", "passMedianMs", "bitsMedianMs", "bitsP10Ms", "bitsP90Ms"]].merge(base[["entries", "plaintextMedianMs"]], on="entries")
t["bits / plaintext"] = t["bitsMedianMs"] / t["plaintextMedianMs"]
print(t.round(2).to_string(index=False))

 entries  passMedianMs  bitsMedianMs  bitsP10Ms  bitsP90Ms  plaintextMedianMs  bits / plaintext
       1         90.01         69.83      65.30      73.13              16.74              4.17
       8         83.83         68.09      64.57      74.49              17.00              4.01
      64         88.78         64.54      63.69      66.79              35.68              1.81


## 3. The slow confirm, and the whole crossing at 64 entries

A crossing pays the bitset pass plus one confirm per candidate. The `expected` column uses `N·E/p` false candidates
plus the three true ones; `observed` uses this run's count.

In [5]:
conf = pd.DataFrame(D["confirm"])
print(conf.round(3).to_string(index=False))
c64 = cfg[cfg["entries"] == 64].iloc[0]
rows = []
for _, c in conf.iterrows():
    exp_c = c64["expectedFalse"] + c64["planted"]
    rows.append({"iterations": c["iterations"], "confirmMs": c["medianMs"],
                 "crossing ms (expected)": c64["bitsMedianMs"] + exp_c * c["medianMs"],
                 "crossing ms (observed)": c64["bitsMedianMs"] + c64["candidates"] * c["medianMs"],
                 "plaintext ms": base[base["entries"] == 64]["plaintextMedianMs"].iloc[0]})
print(pd.DataFrame(rows).round(2).to_string(index=False))

 iterations  medianMs  p90Ms
       1000     0.107  0.226
       5000     0.416  0.524
      20000     1.592  1.706
 iterations  confirmMs  crossing ms (expected)  crossing ms (observed)  plaintext ms
     1000.0       0.11                   66.93                   65.92         35.68
     5000.0       0.42                   73.87                   69.95         35.68
    20000.0       1.59                  100.25                   85.23         35.68


## 4. One filter hit against the pass's spread

A text that holds a stored 8-unit prefix pays one more confirm than a text that does not. Telling the two apart is a
difference of two means: with `n` crossings of each, the difference's standard error is `sd·sqrt(2/n)`, so a shift of
one hit is seen at `z` standard errors once `n` reaches `2·(z·sd / hit)²`; here `z = 2` (about 95% on one side). `sd`
is estimated from the bitset pass's 15 repeats as `(p90 - p10) / 2.56` (a normal approximation), so it reflects how
busy the machine was during each configuration's repeats; every configuration's spread is printed, the quiet ones
with the noisy one. This measures only the worker's side: whether the
difference survives to a PAGE through `forPage` (a runtime message, the page's own timer resolution, other work in the
worker) is not measured here and needs an end-to-end run in the extension.

In [6]:
Z = 2.0
rows = []
for _, c in cfg.iterrows():
    spread = c["bitsP90Ms"] - c["bitsP10Ms"]
    sd = spread / 2.56
    for _, k in conf.iterrows():
        rows.append({"entries": c["entries"], "pass spread p90-p10, ms": spread, "sd, ms": sd, "iterations": k["iterations"],
                     "one hit, ms": k["medianMs"], "hit / spread": k["medianMs"] / spread,
                     "crossings of each, z=2": max(1, round(2 * (Z * sd / k["medianMs"]) ** 2))})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 entries  pass spread p90-p10, ms  sd, ms  iterations  one hit, ms  hit / spread  crossings of each, z=2
     1.0                    7.829   3.058      1000.0        0.107         0.014                    6566
     1.0                    7.829   3.058      5000.0        0.416         0.053                     432
     1.0                    7.829   3.058     20000.0        1.592         0.203                      30
     8.0                    9.917   3.874      1000.0        0.107         0.011                   10535
     8.0                    9.917   3.874      5000.0        0.416         0.042                     693
     8.0                    9.917   3.874     20000.0        1.592         0.160                      47
    64.0                    3.092   1.208      1000.0        0.107         0.035                    1024
    64.0                    3.092   1.208      5000.0        0.416         0.135                      67
    64.0                    3.092   1.208     20000.0  

## What this says, and what it does not

- **The forms count was too high in the spec.** About five forms a secret, `E` ≈ 313 at 64 entries (`T` ≈ 278
  distinct filters), so about 20 false candidates a MiB at `w` ≈ 24, not 44: the model `N·E/p` holds in the mean on
  random text and base64, and on repetitive text (JSON, logs) the count is overdispersed, which the per-window memo
  bounds (section 1b).
- **The filter pass is the cost, not the confirm.** The bitset pass costs a small multiple of the plaintext scrub over
  the same MiB (section 2); the confirms add little at 20,000 iterations (section 3). A crossing at 64 entries costs about 65 to 100 ms a MiB on this run, the top at 20,000 iterations, on a loaded machine.
- **The timing oracle is real on the worker's side.** On this run's shared, loaded machine one confirm sits inside a
  single crossing's spread, and a difference of means over repeated crossings separates it after the number in
  section 4's last column: from a handful to a few dozen crossings at 20,000 iterations, one to ten thousand at 1,000. That number scales with
  `sd²`, so a quieter machine needs fewer; a slower confirm is the louder one. Whether a page can
  repeat crossings and time them through `forPage` precisely enough is the open measurement; until it is made, the
  spec's remedy (a fixed confirm schedule, or a floor on a page's crossing) should be assumed needed.
- **Not measured here:** the confirm's resistance to guessing (that is the iteration count's job, and the spec's cost
  paragraph says what a stolen store gives), a hostile text built to collide filters (the prime modulus and the secret
  base are what stand against it), and another machine.